# Phase 8 — Evaluation & Experiments Framework

An interactive educational walkthrough of **Phase 8: Evaluation & Experiments (`v0.8.0`)** for Motor Imagery EEG Classification.
This notebook demonstrates checkpoint loading, manifest generation (`manifest.json`), multi-metric evaluation (Accuracy, Balanced Accuracy, Cohen's Kappa, Macro/Weighted F1, Per-class metrics), visualization (Confusion Matrix Heatmap, 2D t-SNE Embedding Projections, ACA Attention Heatmaps), raw predictions serialization (`raw_predictions.pt`), and comparative architectural ablation matrix execution.

## 1. Objective & Scientific Evaluation Pipeline

Phase 8 establishes a publishable research evaluation pipeline for trained checkpoints (`.pt`), ensuring complete evaluation reproducibility:

```text
Checkpoint (.pt)
       │
       ▼  [EvaluationRunner / manifest.json]
InferenceEngine (torch.no_grad)
       │
       ▼  [Predictions, CLS Embeddings, ACA Attention Weights]
Evaluator
 ┌─────┼────────────────────────────┐
 ▼     ▼                            ▼
Metrics Visualization          Report Generator & Raw Serialization
 (Acc, F1,  (Confusion Matrix,  (evaluation_report.json, metrics.csv,
  Kappa)     t-SNE Embeddings)   predictions.csv, raw_predictions.pt)
```

## 2. Implementation Imports & Configuration Setup

In [ ]:
import os
import sys
import torch
import pandas as pd
import matplotlib.pyplot as plt

def get_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr and not os.path.exists(os.path.join(curr, "models")):
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent
    return curr

PROJECT_ROOT = get_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"[OK] Project Root set to: {PROJECT_ROOT}")

from configs.config_loader import load_master_config
from models.eeg_motor_imagery_model import EEGMotorImageryModel
from datasets.builder import build_dataloaders
from evaluation import Evaluator, InferenceEngine, EmbeddingProjector, compute_summary_statistics
from visualization import plot_confusion_matrix, plot_embedding_projection, plot_attention_heatmap
from experiments.ablation import AblationRunner

master_config = load_master_config(project_root=PROJECT_ROOT)
print("[OK] Successfully imported Evaluation & Experimentation modules.")

## 3. End-to-End Evaluation & Report Generation

Running `Evaluator` on a trained checkpoint, calculating metrics, exporting reports, and saving raw PyTorch prediction tensors.

In [ ]:
# Set up model & evaluation loader
model = EEGMotorImageryModel.from_config(master_config)
ckpt_path = os.path.join(PROJECT_ROOT, "outputs", "checkpoints", "latest.pt")
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path)
    model.load_state_dict(ckpt.get("model_state", ckpt))
model.eval()

_, val_loader, _ = build_dataloaders(master_config)
out_dir = os.path.join(PROJECT_ROOT, "outputs", "evaluation")

evaluator = Evaluator(model=model, output_dir=out_dir)
metrics, results = evaluator.evaluate(
    dataloader=val_loader,
    config=master_config,
    checkpoint_name="latest.pt",
    generate_plots=True,
)

print("\n=== Evaluator Metric Results ===")
print(f"Accuracy:          {metrics['accuracy']:.4f}")
print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
print(f"Macro F1:          {metrics['f1']:.4f}")
print(f"Weighted F1:       {metrics['weighted_f1']:.4f}")
print(f"Cohen's Kappa:     {metrics['cohen_kappa']:.4f}")

## 4. Visualizations

### 4.1 Confusion Matrix Heatmap

In [ ]:
class_names = ["Left Hand", "Right Hand", "Both Hands", "Feet"]
fig_cm = plot_confusion_matrix(metrics["confusion_matrix"], class_names=class_names)
plt.show()

### 4.2 2D t-SNE Projection of Global CLS Embeddings

In [ ]:
projector = EmbeddingProjector(seed=42)
cls_np = results.cls_embeddings.numpy()
targets_np = results.targets.numpy()

emb_2d = projector.project_tsne(cls_np)
fig_tsne = plot_embedding_projection(emb_2d, targets_np, class_names=class_names, method="t-SNE")
plt.show()

### 4.3 Adaptive Channel Attention Weight Heatmap

In [ ]:
if results.attention_weights is not None:
    att_np = results.attention_weights.numpy()
    fig_att = plot_attention_heatmap(att_np, band_names=["Theta", "Alpha", "Beta", "Gamma"])
    plt.show()

## 5. Architectural Ablation Matrix Execution

Evaluating performance across model variants (Baseline, Frequency Only, + ACA, + FATE Full Model).

In [ ]:
ablation_runner = AblationRunner(output_dir=out_dir)
df_ablation = ablation_runner.run_ablation(dataloader=val_loader)

print("\n=== Ablation Matrix Summary ===")
display(df_ablation.style.format({
    "Accuracy": "{:.4f}", "Balanced Accuracy": "{:.4f}", "Macro F1": "{:.4f}", "Cohen Kappa": "{:.4f}"
}))

## 6. Exported Artifact Verification

Listing all exported evaluation artifacts in `outputs/evaluation/`.

In [ ]:
files = os.listdir(out_dir)
print(f"Exported Evaluation Files in {out_dir}:")
for f in sorted(files):
    f_path = os.path.join(out_dir, f)
    if os.path.isfile(f_path):
        size_kb = os.path.getsize(f_path) / 1024.0
        print(f"  - {f:<25} ({size_kb:.2f} KB)")

## 7. Conclusion & Research Roadmap

### Key Takeaways:
1. **Traceable Manifest**: Automatically generates `manifest.json` with git commit hash, model version `v0.8.0`, PyTorch/CUDA info, and config hash.
2. **Comprehensive Metric Suite**: Computes Accuracy, Balanced Accuracy, Cohen's Kappa, Macro/Weighted F1, and Per-class metrics.
3. **Raw Tensor Serialization**: Saves `raw_predictions.pt` to allow figure regeneration without rerunning model inference.
4. **Visual Analysis**: Seaborn/Matplotlib heatmaps for confusion matrix & attention weights; 2D t-SNE scatter plots for global CLS embeddings.
5. **Ablation Framework**: `AblationRunner` quantifies component contributions across architectural variants.

**Phase 8 is complete and fully validated.** The framework is now ready for **Phase 9 (Hyperparameter Optimization)**.